In [1]:
import pandas as pd
import numpy as np
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.preprocessing import normalize
import re
from nltk.corpus import stopwords

In [2]:
# Load data
abstracts = pd.read_csv('abstracts.txt', delimiter='\t', header=None, names=['abstract'])
authors = pd.read_csv('authors.txt', delimiter='\t', header=None, names=['authors'])
edgelist = pd.read_csv('edgelist.txt', delimiter=',', header=None, names=['source', 'target'])
test_edges = pd.read_csv('test.txt', delimiter=',', header=None, names=['source', 'target'])

In [3]:
# Text preprocessing
def clean_text(text):
    stop_words = set(stopwords.words('english'))
    text = re.sub(r'[^a-zA-Z]', ' ', text).lower()
    tokens = [word for word in text.split() if word not in stop_words]
    return ' '.join(tokens)

abstracts['cleaned'] = abstracts['abstract'].apply(clean_text)

In [5]:
# Convert abstracts to Doc2Vec vectors
documents = [TaggedDocument(doc.split(), [i]) for i, doc in enumerate(abstracts['cleaned'])]
model = Doc2Vec(documents, vector_size=128, window=5, min_count=2, workers=6)

abstracts['vector'] = abstracts['cleaned'].apply(lambda x: model.infer_vector(x.split()))

In [7]:
# Prepare training data from edgelist
positive_pairs = edgelist.copy()
positive_pairs['label'] = 1

In [17]:
negative_pairs = positive_pairs.copy()

# Shuffle the 'target' column to break the correct pairs
negative_pairs['target'] = np.random.permutation(negative_pairs['target'])

# Add labels to both datasets
positive_pairs['label'] = 1
negative_pairs['label'] = 0

# Combine and shuffle
training_data = pd.concat([positive_pairs, negative_pairs]).sample(frac=1).reset_index(drop=True)


In [19]:
# Extract feature vectors (cosine similarity between vectors)
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

training_data['similarity'] = training_data.apply(lambda row: cosine_similarity(
    abstracts['vector'][row['source']], abstracts['vector'][row['target']]
), axis=1)

In [21]:
# Train logistic regression model
X_train, X_val, y_train, y_val = train_test_split(
    training_data[['similarity']], training_data['label'], test_size=0.2, random_state=42
)

lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)

LogisticRegression()

In [23]:
# Evaluate model
val_preds = lr_model.predict_proba(X_val)[:, 1]
print('Validation Log Loss:', log_loss(y_val, val_preds))

Validation Log Loss: 0.553353045221733


In [25]:
# Generate predictions for the test file
test_edges['similarity'] = test_edges.apply(lambda row: cosine_similarity(
    abstracts['vector'][row['source']], abstracts['vector'][row['target']]
), axis=1)

In [27]:
# Predict probabilities
predictions = lr_model.predict_proba(test_edges[['similarity']])[:, 1]

In [39]:
# Save to submission.csv
submission = pd.DataFrame({'ID': range(len(test_edges)),
                           'Label': predictions})
submission.to_csv('submission.csv', index=False)

print("Submission file saved as submission.csv")

Submission file saved as submission.csv
